In [5]:
import requests
import json
import nflreadpy as nfl
import polars as pl

In [2]:
BASE_URL = "https://api.sleeper.app/v1"

# Retrieve all NFL players in Sleeper's player directory
print("Downloading Sleeper player data...")

response = requests.get(
    f"{BASE_URL}/players/nfl",
    timeout=60
)

response.raise_for_status()
players = response.json()

print(f"\nTotal players: {len(players):,}")

# Inspect a few players
print("\nSAMPLE PLAYERS")

sample_players = list(players.items())[:5]

for player_id, player in sample_players:
    print(f"\nSleeper ID: {player_id}")
    print(json.dumps(player, indent=2)[:1500])

# Inspect available fields
print("\nAVAILABLE PLAYER FIELDS")

fields = set()

for player in players.values():
    fields.update(player.keys())

for field in sorted(fields):
    print(field)

# Check player ID mapping coverage
print("\nPLAYER ID COVERAGE")

gsis_count = sum(
    1 for p in players.values()
    if p.get("gsis_id")
)

print(f"Players with GSIS IDs: {gsis_count:,}")
print(f"Players without GSIS IDs: {len(players) - gsis_count:,}")

# Check positions
print("\nPOSITION COUNTS")

position_counts = {}

for player in players.values():
    position = player.get("position") or "Unknown"
    position_counts[position] = position_counts.get(position, 0) + 1

for position, count in sorted(position_counts.items()):
    print(f"{position}: {count:,}")


Total players: 12,229

SAMPLE PLAYERS

Sleeper ID: 6462
{
  "full_name": "Ellis Richardson",
  "swish_id": null,
  "gsis_id": " 00-0035057",
  "rotoworld_id": null,
  "competitions": [],
  "rotowire_id": 14134,
  "stats_id": 887183,
  "hashtag": "#ellisrichardson-NFL-FA-45",
  "fantasy_data_id": 21427,
  "college": "Georgia Southern",
  "injury_status": null,
  "active": true,
  "metadata": {
    "channel_id": "1116853254748110848"
  },
  "birth_date": "1995-02-12",
  "number": 45,
  "news_updated": null,
  "high_school": "Douglas County",
  "yahoo_id": 32262,
  "sportradar_id": "efd6f3c3-b752-4bc2-a4f2-b776c15c3ec0",
  "height": "75",
  "oddsjam_id": null,
  "search_full_name": "ellisrichardson",
  "status": "Active",
  "injury_start_date": null,
  "birth_country": null,
  "team": null,
  "kalshi_id": null,
  "team_abbr": null,
  "birth_state": null,
  "practice_description": null,
  "injury_notes": null,
  "team_changed_at": null,
  "fantasy_positions": [
    "TE"
  ],
  "depth_char

In [6]:


# 1. Create Sleeper GSIS ID set
gsis_ids = {
    player["gsis_id"].strip()
    for player in players.values()
    if player.get("gsis_id") and player["gsis_id"].strip()
}

# 2. Load NFLverse players
nfl_players = nfl.load_players()

# 3. Create NFLverse GSIS ID set
nfl_gsis_ids = set(
    nfl_players
    .select(
        pl.col("gsis_id")
        .drop_nulls()
        .str.strip_chars()
    )
    .to_series()
    .to_list()
)

# 4. Compare IDs
matched = gsis_ids.intersection(nfl_gsis_ids)

print("\nGSIS ID MATCHING RESULTS")
print(f"Sleeper unique GSIS IDs: {len(gsis_ids):,}")
print(f"NFLverse unique GSIS IDs: {len(nfl_gsis_ids):,}")
print(f"Matching IDs: {len(matched):,}")
print(f"Sleeper IDs without NFLverse match: {len(gsis_ids - nfl_gsis_ids):,}")

if gsis_ids:
    print(f"Match percentage: {len(matched) / len(gsis_ids) * 100:.2f}%")


GSIS ID MATCHING RESULTS
Sleeper unique GSIS IDs: 3,886
NFLverse unique GSIS IDs: 24,864
Matching IDs: 3,525
Sleeper IDs without NFLverse match: 361
Match percentage: 90.71%


In [7]:
# Analyze Sleeper players without GSIS IDs

missing_gsis = [
    player for player in players.values()
    if not player.get("gsis_id")
    or not player.get("gsis_id").strip()
]

print("\nSLEEPER PLAYERS WITHOUT GSIS IDs")
print(f"Total: {len(missing_gsis):,}")

# Fantasy-relevant positions
fantasy_positions = {"QB", "RB", "WR", "TE", "K"}

relevant_players = [
    p for p in missing_gsis
    if any(
        pos in fantasy_positions
        for pos in (p.get("fantasy_positions") or [])
    )
]

print(f"Fantasy-relevant: {len(relevant_players):,}")

# Breakdown by fantasy position
position_counts = {}

for player in relevant_players:
    for position in player.get("fantasy_positions") or []:
        if position in fantasy_positions:
            position_counts[position] = (
                position_counts.get(position, 0) + 1
            )

print("\nFANTASY POSITION COUNTS")

for position, count in sorted(position_counts.items()):
    print(f"{position}: {count:,}")

# Check alternative ESPN identifiers
espn_ids = [
    p for p in missing_gsis
    if p.get("espn_id") is not None
]

print(f"\nPlayers with ESPN IDs: {len(espn_ids):,}")

# Check players currently assigned to an NFL team
on_team = [
    p for p in missing_gsis
    if p.get("team")
]

print(f"Players assigned to a team: {len(on_team):,}")

# Check active status
active = [
    p for p in missing_gsis
    if p.get("active") is True
]

print(f"Players marked active: {len(active):,}")


SLEEPER PLAYERS WITHOUT GSIS IDs
Total: 8,336
Fantasy-relevant: 2,984

FANTASY POSITION COUNTS
K: 130
QB: 324
RB: 708
TE: 567
WR: 1,261

Players with ESPN IDs: 2,856
Players assigned to a team: 2,197
Players marked active: 6,297


In [8]:
# Load NFLverse player directory
nfl_players = nfl.load_players()

print("NFLVERSE PLAYER COLUMNS")
print(nfl_players.columns)

# Check whether ESPN IDs are available
if "espn_id" in nfl_players.columns:

    # Create NFLverse ESPN ID set
    nfl_espn_ids = set(
        nfl_players
        .select("espn_id")
        .drop_nulls()
        .to_series()
        .cast(pl.String)
        .to_list()
    )

    # Find fantasy-relevant Sleeper players without GSIS IDs
    missing_gsis = [
        (sleeper_id, player)
        for sleeper_id, player in players.items()
        if not (player.get("gsis_id") or "").strip()
        and any(
            pos in {"QB", "RB", "WR", "TE", "K"}
            for pos in (player.get("fantasy_positions") or [])
        )
    ]

    # Check ESPN matches
    matched_espn = []

    for sleeper_id, player in missing_gsis:
        espn_id = player.get("espn_id")

        if espn_id is not None:
            if str(espn_id) in nfl_espn_ids:
                matched_espn.append({
                    "sleeper_id": sleeper_id,
                    "name": player.get("full_name"),
                    "espn_id": espn_id,
                    "position": player.get("position")
                })

    print("\nESPN MATCHING RESULTS")
    print(f"Fantasy players without GSIS: {len(missing_gsis):,}")
    print(f"Players with ESPN matches: {len(matched_espn):,}")
    print(f"Players without ESPN matches: {len(missing_gsis) - len(matched_espn):,}")

else:
    print("NFLverse does not contain an espn_id column.")

NFLVERSE PLAYER COLUMNS
['gsis_id', 'display_name', 'common_first_name', 'first_name', 'last_name', 'short_name', 'football_name', 'suffix', 'esb_id', 'nfl_id', 'pfr_id', 'pff_id', 'otc_id', 'espn_id', 'smart_id', 'birth_date', 'position_group', 'position', 'ngs_position_group', 'ngs_position', 'height', 'weight', 'headshot', 'college_name', 'college_conference', 'jersey_number', 'rookie_season', 'last_season', 'latest_team', 'status', 'ngs_status', 'ngs_status_short_description', 'years_of_experience', 'pff_position', 'pff_status', 'draft_year', 'draft_round', 'draft_pick', 'draft_team']

ESPN MATCHING RESULTS
Fantasy players without GSIS: 2,984
Players with ESPN matches: 783
Players without ESPN matches: 2,201


In [9]:
# Normalize ESPN IDs
def normalize_id(value):
    if value is None:
        return None

    value = str(value).strip()

    if not value or value.lower() == "nan":
        return None

    return value


# Build NFLverse ESPN ID mapping
nfl_espn_map = {}

for player in nfl_players.to_dicts():
    espn_id = normalize_id(player.get("espn_id"))

    if espn_id:
        nfl_espn_map.setdefault(espn_id, []).append(player)


# Find Sleeper players without GSIS IDs
sleeper_missing_gsis = []

for sleeper_id, player in players.items():
    if (
        not normalize_id(player.get("gsis_id"))
        and any(
            pos in {"QB", "RB", "WR", "TE", "K"}
            for pos in (player.get("fantasy_positions") or [])
        )
    ):
        sleeper_missing_gsis.append((sleeper_id, player))


# Identify duplicate ESPN IDs in NFLverse
duplicate_nfl_espn = {
    espn_id: records
    for espn_id, records in nfl_espn_map.items()
    if len(records) > 1
}

# Identify duplicate ESPN IDs in Sleeper subset
sleeper_espn_counts = {}

for sleeper_id, player in sleeper_missing_gsis:
    espn_id = normalize_id(player.get("espn_id"))

    if espn_id:
        sleeper_espn_counts.setdefault(espn_id, []).append(
            (sleeper_id, player)
        )

duplicate_sleeper_espn = {
    espn_id: records
    for espn_id, records in sleeper_espn_counts.items()
    if len(records) > 1
}


# Display results
print("\nESPN ID QUALITY CHECK")
print(f"NFLverse unique ESPN IDs: {len(nfl_espn_map):,}")
print(f"NFLverse duplicate ESPN IDs: {len(duplicate_nfl_espn):,}")
print(f"Sleeper duplicate ESPN IDs: {len(duplicate_sleeper_espn):,}")

# Show actual matching players
print("\nSAMPLE ESPN MATCHES")

match_count = 0

for sleeper_id, player in sleeper_missing_gsis:
    espn_id = normalize_id(player.get("espn_id"))

    if espn_id in nfl_espn_map:
        nfl_matches = nfl_espn_map[espn_id]

        for nfl_player in nfl_matches:
            print(
                f"Sleeper: {player.get('full_name')} | "
                f"NFLverse: {nfl_player.get('display_name')} | "
                f"ESPN ID: {espn_id}"
            )

        match_count += 1

        if match_count >= 15:
            break


ESPN ID QUALITY CHECK
NFLverse unique ESPN IDs: 16,596
NFLverse duplicate ESPN IDs: 0
Sleeper duplicate ESPN IDs: 0

SAMPLE ESPN MATCHES
Sleeper: Tyler Slavin | NFLverse: Tyler Slavin | ESPN ID: 2516911
Sleeper: Lance Dunbar | NFLverse: Lance Dunbar | ESPN ID: 15369
Sleeper: Taylor Thompson | NFLverse: Taylor Thompson | ESPN ID: 15067
Sleeper: Johnny Manziel | NFLverse: Johnny Manziel | ESPN ID: 16736
Sleeper: Salvon Ahmed | NFLverse: Salvon Ahmed | ESPN ID: 4243315
Sleeper: Ryan Whalen | NFLverse: Ryan Whalen | ESPN ID: 14131
Sleeper: CeeDee Lamb | NFLverse: CeeDee Lamb | ESPN ID: 4241389
Sleeper: Jerry Jeudy | NFLverse: Jerry Jeudy | ESPN ID: 4241463
Sleeper: Knowshon Moreno | NFLverse: Knowshon Moreno | ESPN ID: 12516
Sleeper: Andrew Peacock | NFLverse: Andrew Peacock | ESPN ID: 17025
Sleeper: Dri Archer | NFLverse: Dri Archer | ESPN ID: 16755
Sleeper: Leonard Hankerson | NFLverse: Leonard Hankerson | ESPN ID: 14021
Sleeper: Chris Pantale | NFLverse: Chris Pantale | ESPN ID: 16210


In [10]:
# Build NFLverse identifier lookup dictionaries
nfl_by_gsis = {}
nfl_by_espn = {}

for player in nfl_players.to_dicts():
    gsis = normalize_id(player.get("gsis_id"))
    espn = normalize_id(player.get("espn_id"))

    if gsis:
        nfl_by_gsis.setdefault(gsis, []).append(player)

    if espn:
        nfl_by_espn.setdefault(espn, []).append(player)


# Check every Sleeper player for identifier conflicts
conflicts = []
consistent = 0
gsis_only = 0
espn_only = 0
unmatched = 0

for sleeper_id, player in players.items():

    gsis = normalize_id(player.get("gsis_id"))
    espn = normalize_id(player.get("espn_id"))

    gsis_matches = nfl_by_gsis.get(gsis, []) if gsis else []
    espn_matches = nfl_by_espn.get(espn, []) if espn else []

    if len(gsis_matches) == 1 and len(espn_matches) == 1:

        gsis_player = gsis_matches[0]
        espn_player = espn_matches[0]

        if normalize_id(gsis_player.get("gsis_id")) == normalize_id(
            espn_player.get("gsis_id")
        ):
            consistent += 1
        else:
            conflicts.append({
                "sleeper_name": player.get("full_name"),
                "gsis_match": gsis_player.get("display_name"),
                "espn_match": espn_player.get("display_name"),
                "sleeper_gsis": gsis,
                "sleeper_espn": espn
            })

    elif len(gsis_matches) == 1:
        gsis_only += 1

    elif len(espn_matches) == 1:
        espn_only += 1

    else:
        unmatched += 1


print("\nIDENTIFIER CONSISTENCY RESULTS")
print(f"Consistent GSIS + ESPN matches: {consistent:,}")
print(f"GSIS-only matches: {gsis_only:,}")
print(f"ESPN-only matches: {espn_only:,}")
print(f"Conflicting matches: {len(conflicts):,}")
print(f"Unmatched or ambiguous: {unmatched:,}")

print("\nCONFLICT EXAMPLES")
for conflict in conflicts[:20]:
    print(conflict)


IDENTIFIER CONSISTENCY RESULTS
Consistent GSIS + ESPN matches: 3,508
GSIS-only matches: 22
ESPN-only matches: 2,165
Conflicting matches: 2
Unmatched or ambiguous: 6,532

CONFLICT EXAMPLES
{'sleeper_name': 'Isaiah Searight', 'gsis_match': 'Quinnen Williams', 'espn_match': 'Isaiah Searight', 'sleeper_gsis': '00-0035718', 'sleeper_espn': '3909346'}
{'sleeper_name': 'Christian Jones', 'gsis_match': 'Chris Jones', 'espn_match': 'Christian Jones', 'sleeper_gsis': '00-0034641', 'sleeper_espn': '17070'}
